In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
from pyhydra.data_sources.river_discharge import (
    read_grdc,
    read_grdc_metadata,
    read_grdc_folder,
    analyze_grdc_quality,
)
import matplotlib.pyplot as plt
import pandas as pd

# 🌊 GRDC — Global Runoff Data Centre

## 📚 General Description

The **Global Runoff Data Centre (GRDC)** is the international repository for river discharge data, operated by the German Federal Institute of Hydrology (BfG). It collects and archives daily and monthly streamflow records from gauging stations worldwide.

---

## 💡 Why pre-downloading is required

GRDC **does not expose a public download API**. Data must be requested through the online portal and is delivered as text files:

1. Go to https://portal.grdc.bafg.de
2. Select stations and variables of interest
3. Submit the data request (usually fulfilled within a few days)
4. Download the `.day` (daily) or `.mon` (monthly) files

Once the files are available locally, `pyhydra` parses them automatically.

---

## 📦 File format

GRDC files are **semicolon-delimited** text files with a metadata header:

```
# GRDC-No.: 1234567
# River: TEST RIVER
# Station: MY STATION
# ...
# YYYY-MM-DD;hh;Original;Calculated;Qualifier
2000-01-01; 00; 45.30; 45.30; G
2000-01-02; 00; 50.10; 50.10; G
```

| Column | Description |
|--------|-------------|
| Date | YYYY-MM-DD |
| Original | Reported discharge (m³/s) |
| Calculated | Quality-controlled discharge (m³/s) |
| Qualifier | Data quality flag (`G`=good, `M`=missing, ...) |

> ℹ️ Values below −990 are GRDC missing-value sentinels and are automatically set to `NaN`.

---
## 1. 🗂️ Reading a single GRDC file

In [ ]:
# === CONFIGURATION ===
GRDC_FILE = "path/to/grdc/file/1234567.day"   # <-- update this path

# --- Station metadata ---
meta = read_grdc_metadata(GRDC_FILE)
print(f"Station   : {meta.get('station')}")
print(f"River     : {meta.get('river')}")
print(f"Country   : {meta.get('country')}")
print(f"GRDC No.  : {meta.get('grdc_no')}")
print(f"Latitude  : {meta.get('latitude'):.3f}°")
print(f"Longitude : {meta.get('longitude'):.3f}°")
if 'altitude_m' in meta:
    print(f"Altitude  : {meta['altitude_m']:.0f} m")
if 'catchment_area_km2' in meta:
    print(f"Area      : {meta['catchment_area_km2']:.0f} km²")

In [ ]:
# --- Read discharge series ---
df = read_grdc(GRDC_FILE).set_index("date")
df.head()

In [ ]:
# --- Quality statistics ---
q = analyze_grdc_quality(df.reset_index())
print(f"Period   : {q['start'].date()} → {q['end'].date()}")
print(f"N days   : {q['n_days']}")
print(f"Missing  : {q['missing_pct']} %")
print(f"Mean Q   : {q['mean_m3s']} m³/s")
print(f"Max  Q   : {q['max_m3s']} m³/s")

In [ ]:
# --- Time series plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df.index, df["discharge"], lw=0.7, color="steelblue")
axes[0].set_ylabel("Discharge (m³/s)")
axes[0].set_title(f"GRDC — {meta.get('station', 'Station')} · {meta.get('river', '')} · {meta.get('country', '')}", fontsize=13)
axes[0].grid(alpha=0.3)

annual = df["discharge"].resample("YE").mean()
axes[1].bar(annual.index.year, annual.values, color="steelblue", alpha=0.8)
axes[1].axhline(annual.mean(), color="red", lw=1.2, ls="--", label=f"Mean: {annual.mean():.1f} m³/s")
axes[1].set_ylabel("Mean annual discharge (m³/s)")
axes[1].set_xlabel("Year")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 📂 Reading an entire folder of GRDC files

In [ ]:
# Read all .day files in a folder
GRDC_FOLDER = "path/to/grdc/folder/"   # <-- update this path

# all_data = read_grdc_folder(GRDC_FOLDER, pattern="*.day")
# print(f"Stations loaded: {len(all_data)}")
# for name, df_station in all_data.items():
#     q = analyze_grdc_quality(df_station)
#     print(f"  {name}: {q['n_days']} days, {q['missing_pct']}% missing, mean={q['mean_m3s']} m³/s")
print("(Update GRDC_FOLDER path to run)")

---
## 3. 📊 Seasonal regime

In [ ]:
monthly_mean = df["discharge"].groupby(df.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly_mean.index, monthly_mean.values, color="steelblue", alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("Mean discharge (m³/s)")
ax.set_title(f"Seasonal regime · {meta.get('station', 'GRDC station')}", fontsize=13)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()